In [17]:
import nbformat

file = "llm.ipynb"  # notebook must exist in same folder

with open(file, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

nb.metadata.pop("widgets", None)

with open(file, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Cleaned:", file)

Cleaned: llm.ipynb


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/customer_support_tickets_200k.csv", on_bad_lines="skip")

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

df['category'].value_counts().plot(kind='bar')
plt.title("Category Distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['category'].value_counts().head(10)

In [ ]:
cols = [
    'issue_description',
    'category',
    'priority',
    'channel',
    'product',
    'language',
    'customer_segment',
    'issue_complexity_score',
    'resolution_time_hours',
    'customer_satisfaction_score'
]

df_clean = df[cols].copy()
df_clean.head()

In [ ]:
df.groupby(['category', 'priority']).size().unstack().fillna(0)

In [ ]:
df.groupby(['category', 'channel']).size().unstack().fillna(0)

In [ ]:
df['language'].value_counts()

In [ ]:
df.groupby('category')['issue_complexity_score'].mean().sort_values(ascending=False)

In [ ]:
df.groupby('category')['resolution_time_hours'].mean().sort_values(ascending=False)

In [ ]:
df.groupby('category')['customer_satisfaction_score'].mean().sort_values()

In [ ]:
df_rag = df[['issue_description', 'category']].dropna().copy()
labels = sorted(df_rag['category'].unique().tolist())

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:

embeddings = embedder.encode(
    df_rag['issue_description'].tolist(),
    show_progress_bar=True
)

df_rag['embedding'] = list(embeddings)

In [ ]:
import numpy as np

def retrieve_similar(query, top_k=3):
    q_vec = embedder.encode(query)

    emb_matrix = np.vstack(df_rag['embedding'].values)

    sims = np.dot(emb_matrix, q_vec) / (
        np.linalg.norm(emb_matrix, axis=1) * np.linalg.norm(q_vec)
    )

    top_idx = sims.argsort()[-top_k:][::-1]

    return df_rag.iloc[top_idx]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
def build_rag_prompt(text):
    examples = retrieve_similar(text)

    context = ""
    for _, row in examples.iterrows():
        context += f"""
Ticket: {row['issue_description']}
Category: {row['category']}
"""

    return f"""
You are a strict classifier for customer support tickets.

Use the examples below:

{context}

Now classify this ticket:
{text}

Rules:
- Output ONLY ONE category
- Must be exactly from allowed categories
- No explanation

Answer:
"""

In [ ]:
def llm_predict(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def rag_llm_predict(text):
    prompt = build_rag_prompt(text)
    return llm_predict(prompt)

In [ ]:
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
from sklearn.model_selection import train_test_split

test_data = df_rag[['issue_description', 'category']].sample(1000, random_state=42)

In [ ]:
def predict_rag(text):
    output = rag_llm_predict(text)

    for label in labels:
        if label.lower() in output.lower():
            return label

    return "Unknown"

In [1]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

y_true = test_data['category'].tolist()
texts = test_data['issue_description'].tolist()

y_pred = []

def safe_predict(text):
    try:
        return predict_rag(text)
    except:
        return "error"

with ThreadPoolExecutor(max_workers=8) as executor:
    y_pred = list(tqdm(
        executor.map(safe_predict, texts),
        total=len(texts)
    ))

NameError: name 'test_data' is not defined

In [2]:
from sklearn.metrics import accuracy_score

print("RAG Accuracy:", accuracy_score(y_true, y_pred))

KeyboardInterrupt: 